# Extended Automated Evaluation

Evaluates both the GRADE and RoB2 chatbots using:

| Category | Metrics |
|---|---|
| **RAGAS (semantic)** | Context Recall, Faithfulness, Factual Correctness, Answer Relevancy |
| **N-gram overlap** | ROUGE-1, ROUGE-2, ROUGE-L, BLEU |
| **Semantic similarity** | Cosine similarity (answer vs. reference embeddings) |
| **NLP complexity** | Readability (Flesch, SMOG, Gunning Fog), lexical diversity, syntactic complexity |
| **G-Eval** | LLM-scored Accuracy, Completeness, Clarity (1–5) |

Run this notebook after running `grade_rag.ipynb` and `rob2_rag.ipynb`.

In [1]:
import os, json, re, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import pandas as pd
import textstat
import spacy
import nltk
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    LLMContextRecall, Faithfulness, FactualCorrectness, AnswerRelevancy
)

from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nlp = spacy.load('en_core_web_sm')

from rag_core import get_embeddings

## 1. Configuration

In [2]:
EVAL_MODEL = os.getenv("EVAL_MODEL_NAME", "google/gemini-2.5-flash")
MODEL_NAME = os.getenv("MODEL_NAME", "unknown")

eval_llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model=EVAL_MODEL,
    temperature=0.0,
)
embeddings = get_embeddings()

print(f"Chat model: {MODEL_NAME}")
print(f"Eval model: {EVAL_MODEL}")

c:\Users\DELL\Desktop\magisterka\dev\rag-grade-rob2\rag_core.py:40: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  _embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chat model: mistralai/mistral-nemo
Eval model: google/gemini-2.5-flash


## 2. Load Datasets and Generate Answers

In [3]:
# GRADE
# grade_df = pd.read_csv("evaluation_details/evaluation_gradebook/full_dataset_gradebook.csv")
grade_df = pd.read_csv("testing_datasets/questions_gradebook.csv", sep=";")
grade_sample = grade_df.sample(min(50, len(grade_df)), random_state=42).reset_index(drop=True)

from grade_rag import ask_grade, get_grade_contexts  # requires grade_rag to have been executed

print("Generating GRADE answers...")
grade_sample["chatbot_answer"] = grade_sample["question"].apply(ask_grade)
grade_sample["retrieved_contexts"] = grade_sample["question"].apply(
    lambda q: get_grade_contexts(q, k=5)
)
print("Done.")

Found 43 GRADE PDFs
Loading index from cache: .faiss_cache/grade
Ready.
Answer:
 The four levels of certainty of evidence in GRADE are:

1. **High**: Further research is very unlikely to change our confidence in the estimate of effect.
2. **Moderate**: Further research is likely to have an important impact on our confidence in the estimate of effect and may change the estimate.
3. **Low**: Further research is very likely to have an important impact on our confidence in the estimate of effect and is likely to change the estimate.
4. **Very Low**: Any estimate of effect is very uncertain.

--- Retrieved contexts ---
[1]  a particular range."
(5,6)
And for evidence from reviews of qualitative research: “an assessment of the extent to which the review finding is a reasonable representation of the phenomenon of interest...
[2] users to make judgments within these domains. Signalling questions, e.g. about the priority of a problem, are used to answer questions for each criterion.
Requirement

In [4]:
# RoB2
rob2_path = "testing_datasets/questions_rob2.csv"
rob2_df = None

if os.path.exists(rob2_path):
    rob2_df = pd.read_csv(rob2_path, sep=";")
    rob2_sample = rob2_df.sample(min(50, len(rob2_df)), random_state=42).reset_index(drop=True)

    from rob2_rag import ask_rob2, get_rob2_contexts  # requires rob2_rag to have been executed

    print("Generating RoB2 answers...")
    rob2_sample["chatbot_answer"] = rob2_sample["question"].apply(ask_rob2)
    rob2_sample["retrieved_contexts"] = rob2_sample["question"].apply(
        lambda q: get_rob2_contexts(q, k=5)
    )
    print("Done.")
else:
    print("RoB2 dataset not found — skipping RoB2 evaluation.")

Found 3 RoB2 PDFs
Cache cleared.
Building index from scratch...
Checking cache directory: data/parsed_rob2
Found 3 JSON files in cache directory.
Found 3 pre-parsed JSON files.
  Parsing PDF: data\rob2_pdfs\1.pdf
  Parsing PDF: data\rob2_pdfs\2.pdf
  Parsing PDF: data\rob2_pdfs\3.pdf
Total documents: 0
Index saved to: .faiss_cache/rob2
Ready.
--- Chunk 1 (source: 1.pdf) ---
Signalling questions and criteria for judging risk of bias
Signalling questions for this domain are provided in Box 11. Criteria for reaching risk-of-bias judgements are given in Table 13, and an algorithm for implementing these is provided in Table 14 and 

--- Chunk 2 (source: 1.pdf) ---
Signalling questions
Inclusion of signalling questions within each domain of bias is a key feature of RoB 2. Signalling questions aim to elicit information relevant to an assessment of risk of bias. They seek to be reasonably factual in nature. Responses to these questions feed into algorithms we have developed to guide users of t

## 3. RAGAS Evaluation

In [5]:
def run_ragas(df, q_col, ref_col, ans_col, ctx_col, eval_llm, label):
    dataset = [
        {
            "user_input": row[q_col],
            "retrieved_contexts": row[ctx_col] if isinstance(row[ctx_col], list) else [row[ctx_col]],
            "response": str(row[ans_col]),
            "reference": str(row[ref_col]),
        }
        for _, row in df.iterrows()
    ]
    evaluation_dataset = EvaluationDataset.from_list(dataset)
    evaluator_llm = LangchainLLMWrapper(eval_llm)

    result = evaluate(
        dataset=evaluation_dataset,
        metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), AnswerRelevancy()],
        llm=evaluator_llm,
    )
    ragas_df = result.to_pandas()
    print(f"\n{label} RAGAS averages:")
    print(ragas_df[["context_recall", "faithfulness", "factual_correctness(mode=f1)", "answer_relevancy"]].mean())
    return ragas_df

model_slug = MODEL_NAME.split("/")[-1]
eval_slug = EVAL_MODEL.split("/")[-1]

grade_ragas = run_ragas(
    grade_sample, "question", "expected_response", "chatbot_answer", "retrieved_contexts",
    eval_llm, "GRADE"
)
grade_ragas.to_csv(f"testing_datasets/testing_gradebook/details_ragas_{model_slug}_{eval_slug}_ext.csv", index=False)

Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


GRADE RAGAS averages:
context_recall                  0.840000
faithfulness                    0.813124
factual_correctness(mode=f1)    0.325918
answer_relevancy                0.944732
dtype: float64


In [6]:
if rob2_df is not None:
    os.makedirs("testing_datasets/testing_rob2", exist_ok=True)
    rob2_ragas = run_ragas(
        rob2_sample, "question", "expected_response", "chatbot_answer", "retrieved_contexts",
        eval_llm, "RoB2"
    )
    rob2_ragas.to_csv(f"testing_datasets/testing_rob2/details_ragas_{model_slug}_{eval_slug}.csv", index=False)

Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


RoB2 RAGAS averages:
context_recall                  0.775667
faithfulness                    0.634587
factual_correctness(mode=f1)    0.459000
answer_relevancy                0.851595
dtype: float64


## 4. ROUGE and BLEU Scores

In [7]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

def compute_ngram_metrics(reference: str, hypothesis: str) -> dict:
    scores = scorer.score(reference, hypothesis)
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smooth) if hyp_tokens else 0.0
    return {
        "rouge1_f": scores['rouge1'].fmeasure,
        "rouge2_f": scores['rouge2'].fmeasure,
        "rougeL_f": scores['rougeL'].fmeasure,
        "bleu": bleu,
    }

def add_ngram_metrics(df, ref_col, ans_col):
    metrics = df.apply(lambda r: compute_ngram_metrics(str(r[ref_col]), str(r[ans_col])), axis=1)
    return pd.concat([df, pd.DataFrame(list(metrics))], axis=1)

grade_sample = add_ngram_metrics(grade_sample, "expected_response", "chatbot_answer")
print("GRADE n-gram averages:")
print(grade_sample[["rouge1_f", "rouge2_f", "rougeL_f", "bleu"]].mean())

if rob2_df is not None:
    rob2_sample = add_ngram_metrics(rob2_sample, "expected_response", "chatbot_answer")
    print("\nRoB2 n-gram averages:")
    print(rob2_sample[["rouge1_f", "rouge2_f", "rougeL_f", "bleu"]].mean())

GRADE n-gram averages:
rouge1_f    0.321254
rouge2_f    0.180297
rougeL_f    0.270212
bleu        0.089024
dtype: float64

RoB2 n-gram averages:
rouge1_f    0.378172
rouge2_f    0.185722
rougeL_f    0.268664
bleu        0.080310
dtype: float64


## 5. Semantic Similarity (Answer vs. Reference)

In [8]:
def compute_semantic_similarity(references: list[str], answers: list[str], emb_model) -> np.ndarray:
    ref_embs = np.array(emb_model.embed_documents(references))
    ans_embs = np.array(emb_model.embed_documents(answers))
    sims = [cosine_similarity([r], [a])[0][0] for r, a in zip(ref_embs, ans_embs)]
    return np.array(sims)

grade_sims = compute_semantic_similarity(
    grade_sample["expected_response"].tolist(), grade_sample["chatbot_answer"].tolist(), embeddings
)
grade_sample["semantic_similarity"] = grade_sims
print(f"GRADE semantic similarity: mean={grade_sims.mean():.3f}, std={grade_sims.std():.3f}")

if rob2_df is not None:
    rob2_sims = compute_semantic_similarity(
        rob2_sample["expected_response"].tolist(), rob2_sample["chatbot_answer"].tolist(), embeddings
    )
    rob2_sample["semantic_similarity"] = rob2_sims
    print(f"RoB2 semantic similarity: mean={rob2_sims.mean():.3f}, std={rob2_sims.std():.3f}")

GRADE semantic similarity: mean=0.713, std=0.163
RoB2 semantic similarity: mean=0.684, std=0.120


## 6. NLP Complexity Metrics (from original evaluation)

In [9]:
def compute_nlp_metrics(text: str) -> dict:
    doc = nlp(text)
    words = [t.text for t in doc if t.is_alpha]
    sentences = list(doc.sents)
    bigrams = list(nltk.bigrams(words))

    sent_texts = [s.text for s in sentences]
    sent_embs = embeddings.embed_documents(sent_texts) if len(sent_texts) > 1 else []
    if len(sent_embs) > 1:
        sims = cosine_similarity(sent_embs)
        avg_sim = (np.sum(sims) - np.trace(sims)) / (sims.shape[0]**2 - sims.shape[0])
        emb_diversity = 1 - avg_sim
    else:
        emb_diversity = 0

    reasoning_kws = {"because", "therefore", "hence", "so", "since", "thus", "if", "then", "reason"}
    depths = [max([len(list(t.ancestors)) for t in s]) for s in sentences] if sentences else [0]

    return {
        "avg_word_length": np.mean([len(w) for w in words]) if words else 0,
        "avg_sentence_length": np.mean([len(s) for s in sentences]) if sentences else 0,
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
        "smog_index": textstat.smog_index(text),
        "gunning_fog": textstat.gunning_fog(text),
        "type_token_ratio": len(set(words)) / len(words) if words else 0,
        "embedding_diversity": emb_diversity,
        "bigram_diversity": len(set(bigrams)) / len(bigrams) if bigrams else 0,
        "reasoning_keyword_ratio": len([w for w in words if w.lower() in reasoning_kws]) / len(words) if words else 0,
        "average_reasoning_steps": sum(1 for w in words if w.lower() in {"if", "then", "so", "thus"}),
        "inference_complexity_score": np.mean(depths),
    }

def add_nlp_metrics(df, ans_col):
    metrics = df[ans_col].apply(compute_nlp_metrics)
    return pd.concat([df, pd.DataFrame(list(metrics))], axis=1)

grade_sample = add_nlp_metrics(grade_sample, "chatbot_answer")
if rob2_df is not None:
    rob2_sample = add_nlp_metrics(rob2_sample, "chatbot_answer")

## 7. G-Eval (LLM-Scored Quality)

In [10]:
GEVAL_PROMPT = """\
Evaluate the following chatbot answer. Score each dimension 1-5.

Question: {question}
Reference Answer: {reference}
Chatbot Answer: {answer}

Dimensions:
- accuracy: Is the answer factually correct? (1=wrong, 5=fully correct)
- completeness: Does it cover all key points? (1=major gaps, 5=complete)
- clarity: Is it well-written and clear? (1=confusing, 5=excellent)

Think step-by-step, then return ONLY valid JSON:
{{"accuracy": <int>, "completeness": <int>, "clarity": <int>, "reasoning": "<one sentence>"}}
"""

def geval_score(question, reference, answer, llm) -> dict:
    prompt = GEVAL_PROMPT.format(
        question=question,
        reference=str(reference)[:800],
        answer=str(answer)[:800],
    )
    try:
        resp = llm.invoke(prompt)
        content = resp.content if hasattr(resp, 'content') else str(resp)
        content = re.sub(r'^```(?:json)?\n?', '', content.strip())
        content = re.sub(r'\n?```$', '', content.strip())
        return json.loads(content)
    except Exception as e:
        return {"accuracy": None, "completeness": None, "clarity": None, "reasoning": str(e)}

def add_geval(df, q_col, ref_col, ans_col, llm):
    scores = df.apply(
        lambda r: geval_score(r[q_col], r[ref_col], r[ans_col], llm), axis=1
    )
    scores_df = pd.DataFrame(list(scores)).add_prefix("geval_")
    return pd.concat([df, scores_df], axis=1)

print("Running G-Eval on GRADE...")
grade_sample = add_geval(grade_sample, "question", "expected_response", "chatbot_answer", eval_llm)

if rob2_df is not None:
    print("Running G-Eval on RoB2...")
    rob2_sample = add_geval(rob2_sample, "question", "expected_response", "chatbot_answer", eval_llm)

Running G-Eval on GRADE...
Running G-Eval on RoB2...


## 8. Merge RAGAS + All Metrics and Save

In [11]:
RAGAS_COLS = ["context_recall", "faithfulness", "factual_correctness(mode=f1)", "answer_relevancy"]

def merge_and_save(sample_df, ragas_df, domain, model_slug, eval_slug, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for col in RAGAS_COLS:
        if col in ragas_df.columns:
            sample_df[col] = ragas_df[col].values

    sample_df.to_csv(f"{out_dir}/full_metrics_{model_slug}_{eval_slug}.csv", index=False)

    numeric_cols = sample_df.select_dtypes(include=np.number).columns.tolist()
    avg_row = sample_df[numeric_cols].mean().to_frame().T
    avg_row.insert(0, "model_name", MODEL_NAME)
    avg_row.insert(1, "domain", domain)

    avg_path = f"{out_dir}/average_metrics.csv"
    if os.path.exists(avg_path):
        avg_row.to_csv(avg_path, mode="a", header=False, index=False)
    else:
        avg_row.to_csv(avg_path, index=False)

    print(f"\n{domain} key averages:")
    key_metrics = [c for c in [
        "context_recall", "faithfulness", "factual_correctness(mode=f1)", "answer_relevancy",
        "rouge1_f", "rougeL_f", "bleu", "semantic_similarity",
        "geval_accuracy", "geval_completeness", "geval_clarity"
    ] if c in sample_df.columns]
    print(sample_df[key_metrics].mean())

merge_and_save(grade_sample, grade_ragas, "GRADE", model_slug, eval_slug,
               "testing_datasets/testing_gradebook")

if rob2_df is not None:
    merge_and_save(rob2_sample, rob2_ragas, "RoB2", model_slug, eval_slug,
                   "testing_datasets/testing_rob2")


GRADE key averages:
context_recall                  0.840000
faithfulness                    0.813124
factual_correctness(mode=f1)    0.325918
answer_relevancy                0.944732
rouge1_f                        0.321254
rougeL_f                        0.270212
bleu                            0.089024
semantic_similarity             0.712898
geval_accuracy                  4.448980
geval_completeness              3.877551
geval_clarity                   4.530612
dtype: float64

RoB2 key averages:
context_recall                  0.775667
faithfulness                    0.634587
factual_correctness(mode=f1)    0.459000
answer_relevancy                0.851595
rouge1_f                        0.378172
rougeL_f                        0.268664
bleu                            0.080310
semantic_similarity             0.684382
geval_accuracy                  4.120000
geval_completeness              3.600000
geval_clarity                   4.360000
dtype: float64
